# Generalized QSP with supplied angles
Planned companion. The five code cells are prepared but unexecuted. They construct a polynomial of an explicit unitary signal; no inverse angle compiler or full Hamiltonian simulation is claimed.


In [ ]:
import sys, importlib.util
from pathlib import Path
assert sys.version_info[:3] == (3, 12, 14)
root = Path.cwd()
script = root / 'generate_simulation_gqsp.py'
if not script.is_file():
    script = root / 'scripts/generate_simulation_gqsp.py'
loader = importlib.util.spec_from_file_location('gqsp_notebook_producer', script)
producer = importlib.util.module_from_spec(loader)
sys.modules[loader.name] = producer
loader.loader.exec_module(producer)
gqsp = producer.gqsp
print('Given-angle prototype; numerical results need this notebook to execute.')
snapshot = producer.source_snapshot()
print('Input source commit:', snapshot['source_commit'])


In [ ]:
case = producer.CASE_VECTOR[1]
spec, rotations = producer.case_spec(case)
print(case)
print('These are supplied rotations, not an inverse phase fit.')


In [ ]:
import matplotlib.pyplot as plt
trial = gqsp.build_measured_trial(spec, rotations, case['input'])
figure = trial.draw('mpl', cregbundle=False)
display(figure)
plt.close(figure)


In [ ]:
report = producer.small_report(spec, rotations, case['input'], snapshot)
p, q = gqsp.polynomial_coefficients(rotations)
print('ascending P coefficients:', p)
print('ascending Q coefficients:', q)
print(report)
print('The tiny comparisons are numerical; physical synthesis and a formal proof are uncomputed.')


In [ ]:
rows = []
for n in (1, 2, 3, 8, 64):
    template = producer.CASE_VECTOR[6]
    local_case = {**template, 'n': n, 'input': 'plus_i' if n == 1 else 'plus_i_x'}
    local_spec, local_rotations = producer.case_spec(local_case)
    generated = gqsp.build_measured_trial(local_spec, local_rotations, local_case['input'])
    rows.append({'n': n, 'degree': len(local_rotations)-1, 'wires': generated.num_qubits, 'instructions': len(generated.data), 'scope': 'generation/counts only'})
print(rows)
print('No dense simulation ran for this size slice; degree is fixed and the unitary signal changes with n.')
producer.recheck_snapshot(snapshot)
